In [4]:
from lxml import etree
from curl_cffi import requests
import time
import random
import json
from pprint import pprint
import os

In [6]:
with requests.Session() as session:
    session.headers.update({
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,text/css,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
    })
    response = session.get('https://cooking.nytimes.com/about-us', impersonate="chrome110")
    tree = etree.HTML(response.text)

In [30]:
sections = tree.xpath('.//article[contains(@class, "authorcard")]')
author_pages = {}

for section in sections:
    link_element = section.xpath('./a')[0]
    current_link = link_element.get('href')
    author_name = link_element.xpath('./p')[0].text

    if current_link:
        current_link = "https://cooking.nytimes.com" + current_link
        author_pages[author_name] = current_link

author_pages

Melissa Clark
Eric Kim
Yewande Komolafe
Genevieve Ko
Kenji López-Alt
Sohla El-Waylly
Claire Saffitz
Priya Krishna
Vaughn Vreeland
Sam Sifton
Emily Weinstein
Tanya Sichynsky
Ali Slagle
Kay Chun
Korsha Wilson
Yotam Ottolenghi
David Tanis
Ligaya Mishan
Lisa Donovan
Alexa Weibel
Christina Morales
Julia Moskin
Kim Severson


{'Melissa Clark': 'https://cooking.nytimes.com/author/melissa-clark',
 'Eric Kim': 'https://cooking.nytimes.com/author/eric-kim',
 'Yewande Komolafe': 'https://cooking.nytimes.com/author/yewande-komolafe',
 'Genevieve Ko': 'https://cooking.nytimes.com/author/genevieve-ko',
 'Kenji López-Alt': 'https://cooking.nytimes.com/author/kenji-lopez-alt',
 'Sohla El-Waylly': 'https://cooking.nytimes.com/author/sohla-el-waylly',
 'Claire Saffitz': 'https://cooking.nytimes.com/author/claire-saffitz',
 'Priya Krishna': 'https://cooking.nytimes.com/author/priya-krishna',
 'Vaughn Vreeland': 'https://cooking.nytimes.com/author/vaughn-vreeland',
 'Sam Sifton': 'https://cooking.nytimes.com/author/sam-sifton',
 'Emily Weinstein': 'https://cooking.nytimes.com/author/emily-weinstein',
 'Tanya Sichynsky': 'https://cooking.nytimes.com/author/tanya-sichynsky',
 'Ali Slagle': 'https://cooking.nytimes.com/author/ali-slagle',
 'Kay Chun': 'https://cooking.nytimes.com/author/kay-chun',
 'Korsha Wilson': 'https:/

In [43]:
def collectLinks(pages):
    loop_counter = 0

    master_links_dictionary = {}
    if os.path.exists('recipe_links.json'):
        with open('recipe_links.json', 'r') as file:
            master_links_dictionary = json.load(file)
            
    for page in pages:
        print(pages[page])

        if page not in master_links_dictionary:
            master_links_dictionary[page] = []
        
        with requests.Session() as session:
            session.headers.update({
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,text/css,*/*;q=0.8",
                "Accept-Language": "en-US,en;q=0.5",
            })
            response = session.get(pages[page], impersonate="chrome110")
            tree = etree.HTML(response.text)
        
        sections = tree.findall('.//a')
        
        for section in sections:
            current_link = section.get('href')
            if current_link and current_link.startswith("/recipe"):
                current_link = "https://cooking.nytimes.com" + current_link
        
                if current_link not in master_links_dictionary[page]:
                    master_links_dictionary[page].append(current_link)
                    
        print("-"*20)
        print(f"Grabbed {len(master_links_dictionary[page])} recipes from {pages[page]}.")
        
        with open('recipe_links.json', 'w') as file:
            json.dump(master_links_dictionary, file, indent=4)
            
        delay = random.uniform(5.0, 20.0)
        print("-"*20)
        print(f"Waiting for {delay:.2f} seconds...")
        
        time.sleep(delay)
        loop_counter += 1

In [44]:
collectLinks(author_pages)

https://cooking.nytimes.com/author/melissa-clark
--------------------
Grabbed 48 recipes from https://cooking.nytimes.com/author/melissa-clark.
--------------------
Waiting for 17.81 seconds...
https://cooking.nytimes.com/author/eric-kim
--------------------
Grabbed 48 recipes from https://cooking.nytimes.com/author/eric-kim.
--------------------
Waiting for 17.14 seconds...
https://cooking.nytimes.com/author/yewande-komolafe
--------------------
Grabbed 48 recipes from https://cooking.nytimes.com/author/yewande-komolafe.
--------------------
Waiting for 5.60 seconds...
https://cooking.nytimes.com/author/genevieve-ko
--------------------
Grabbed 48 recipes from https://cooking.nytimes.com/author/genevieve-ko.
--------------------
Waiting for 7.27 seconds...
https://cooking.nytimes.com/author/kenji-lopez-alt
--------------------
Grabbed 48 recipes from https://cooking.nytimes.com/author/kenji-lopez-alt.
--------------------
Waiting for 7.06 seconds...
https://cooking.nytimes.com/author/

In [46]:
def getPageOfComments(offset, recipe_url):
    
    api_url = "https://www.nytimes.com/svc/community/V3/requestHandler"
    comment_data = {}
    params = {
        "cmd": "GetCommentsAll", 
        "offset": offset,
        "limit": "25",
        "sort": "reader",
        "url": recipe_url
    }
    
    with requests.Session() as session:
        response = session.get(api_url, params=params, impersonate="chrome110")
        
        if response.status_code == 200:
            comments_page = response.json()

    return comments_page

def getRawCommentsFromPage(offset, recipe_url):
    comments_page = getPageOfComments(offset, recipe_url)
    results = comments_page.get('results', {})
    raw_comments = results.get('comments', [])
    return raw_comments

def collectAllComments(recipe_url):
    loop_counter = 0
    offset = 0
    last_page = False
    cleaned_comments = []
    raw_comments = getRawCommentsFromPage(offset, recipe_url)

    while len(raw_comments) != 0:

        print("-"*20)
        print(f"Currently working on page {loop_counter} of the comments. There are {len(raw_comments)} comments on this page.")
        for comment in raw_comments:
            comment_entry = {
                'commentBody' : comment.get('commentBody'),
                'commentType' : comment.get('commentType'),
                'userID' : comment.get('userID'),
                'recommendations' : comment.get('recommendations'),
            }
            
            cleaned_comments.append(comment_entry)
            
            replies = comment.get('replies', {})
            for reply in replies:
                reply_entry = {
                    'commentBody' : reply.get('commentBody'),
                    'commentType' : reply.get('commentType'),
                    'userID' : reply.get('userID'),
                    'recommendations' : reply.get('recommendations'),
                }
                cleaned_comments.append(reply_entry)

        if not last_page:
            delay = random.uniform(5.0, 20.0)
            print("-"*20)
            print(f"Waiting for {delay:.2f} seconds...")
            
            time.sleep(delay)
            loop_counter = loop_counter + 1
    
            if loop_counter % 15 == 0:
                long_pause = random.uniform(45.0, 120.0)
                print(f"Taking a long break for {long_pause/60:.1f} minutes...")
                time.sleep(long_pause)

        offset += 25
        raw_comments = getRawCommentsFromPage(offset, recipe_url)
        if len(raw_comments) < 25:
            last_page = True
        
    return cleaned_comments

In [47]:
def scrapeRecipes(recipe_links, file_name):
    dictionary = []
    loop_counter = 0

    for recipe_link in recipe_links:
        with requests.Session() as session:
            session.headers.update({
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,text/css,*/*;q=0.8",
                "Accept-Language": "en-US,en;q=0.5",
            })
            response = session.get(recipe_link, impersonate="chrome110")
            tree = etree.HTML(response.text)
    
        title_element = tree.find(".//title")
        if title_element is not None:
            title = title_element.text
        else:
            title = "No title found"
            
        print("-"*20)
        print(f"Found a recipe: {title}")
        
        h2s = tree.findall('.//h2')
        h2s
        for h2 in h2s:
            if h2 is not None:
                author_links = h2.xpath('./a')
                for author_link in author_links:
                    author = author_link.text
            else:
                author = h2.text
                
        print("-"*20)
        print(f"Found a recipe author: {author}")
    
        stats_table = tree.xpath('.//dl[contains(@class, "stats_statsTable")]')
        stats = {}
        
        if stats_table:
            all_dts = stats_table[0].xpath('.//dt')
            all_dds = stats_table[0].xpath('.//dd')
        
            for dt, dd in zip(all_dts, all_dds):
                key = dt.xpath('normalize-space(.)')
                value = dd.xpath('normalize-space(.)')
        
                if key != "Comments":
                    stats[key] = value
    
        print("-"*20)
        print(f"Found stats: {stats}")
        
        comments = collectAllComments(recipe_link)
        
        dictionary.append({
            "url" : recipe_link,
            "title" : title,
            "author" : author,
            "stats" : stats,
            "comments" : comments,
        })
    
        with open(f"recipe_data_{file_name}.json", "w", encoding="utf-8") as file:
            json.dump(dictionary, file, indent=4)
    
        delay = random.uniform(5.0, 20.0)
        print("-"*20)
        print(f"Waiting for {delay:.2f} seconds...")
    
        time.sleep(delay)
        loop_counter += 1
    
        if loop_counter % 15 == 0:
            long_pause = random.uniform(120.0, 300.0)
            print(f"Taking a long break for {long_pause/60:.1f} minutes...")
            time.sleep(long_pause)

In [48]:
links_to_scrape = {}
if os.path.exists('recipe_links.json'):
    with open('recipe_links.json', 'r') as file:
        links_to_scrape = json.load(file)
links_to_scrape['Melissa Clark']

['https://cooking.nytimes.com/recipes/782914412-cabbage-caesar-salad',
 'https://cooking.nytimes.com/recipes/781620156-melissa-clarks-best-buttermilk-pancakes',
 'https://cooking.nytimes.com/recipes/780934238-cold-peanut-ginger-noodles',
 'https://cooking.nytimes.com/recipes/779298494-roasted-rhubarb',
 'https://cooking.nytimes.com/recipes/778097361-lemon-chicken-with-asparagus',
 'https://cooking.nytimes.com/recipes/777656134-roasted-sausages-with-chickpeas-and-spinach',
 'https://cooking.nytimes.com/recipes/1023880-vegetarian-tamale-pie',
 'https://cooking.nytimes.com/recipes/776890616-highlander-shortbreads',
 'https://cooking.nytimes.com/recipes/776372620-easy-soft-pretzels',
 'https://cooking.nytimes.com/recipes/774540571-cauliflower-kugel',
 'https://cooking.nytimes.com/recipes/774457175-roasted-carrots-with-harissa-vinaigrette',
 'https://cooking.nytimes.com/recipes/774456092-homemade-matzo',
 'https://cooking.nytimes.com/recipes/1027726-chewy-marmalade-oatmeal-cookies',
 'https

In [49]:
scrapeRecipes(links_to_scrape['Melissa Clark'], "Melissa_Clark")

--------------------
Found a recipe: Cabbage Caesar Salad Recipe
--------------------
Found a recipe author: Melissa Clark
--------------------
Found stats: {'Ready In': '30 min', 'Rating': '5(62)'}
--------------------
Currently working on page 0 of the comments. There are 25 comments on this page.
--------------------
Waiting for 12.56 seconds...
--------------------
Currently working on page 1 of the comments. There are 20 comments on this page.
--------------------
Waiting for 12.81 seconds...
--------------------
Found a recipe: No title found
--------------------
Found a recipe author: Melissa Clark
--------------------
Found stats: {}
--------------------
Currently working on page 0 of the comments. There are 25 comments on this page.
--------------------
Waiting for 16.79 seconds...
--------------------
Currently working on page 1 of the comments. There are 25 comments on this page.
--------------------
Waiting for 5.16 seconds...
--------------------
Currently working on page 